##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

In [23]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


In [24]:

# -----------------------------
# 1) Load CIFAR-10
# -----------------------------
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

class_names = [
    "airplane","automobile","bird","cat","deer",
    "dog","frog","horse","ship","truck"
]

# Keep labels as integers (SparseCategoricalCrossentropy)
y_train = y_train.squeeze().astype("int64")
y_test  = y_test.squeeze().astype("int64")

# Convert images to float32
x_train = x_train.astype("float32")
x_test  = x_test.astype("float32")


In [25]:
# -----------------------------
#Load MobileNetV2 
# -----------------------------
mobilenet_base = MobileNetV2(
    weights="imagenet",
    input_shape=(224, 224, 3),
    include_top=False,
)

In [26]:
# -----------------------------
#Fine-tune last layers
# -----------------------------
mobilenet_base.trainable = True
for layer in mobilenet_base.layers[:-30]:
    layer.trainable = False

print("Trainable layers in backbone:", sum(l.trainable for l in mobilenet_base.layers), "/", len(mobilenet_base.layers))

mobilenet_model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Resizing(224, 224, interpolation="bilinear"),
    layers.Lambda(preprocess_input),          # IMPORTANT
    mobilenet_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10)
])
mobilenet_model.summary()

mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history_ft = mobilenet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=64,
    verbose=1
)

test_loss_ft, test_acc_ft = mobilenet_model.evaluate(x_test, y_test, verbose=0)
print("MobileNet (fine-tuned) test accuracy:", test_acc_ft)
print("MobileNet (fine-tuned) test loss    :", test_loss_ft)

Trainable layers in backbone: 30 / 154


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resizing_2 (Resizing)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_5 (Lambda)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_8      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 1,539,210 (5.87 MB)

 Non-trainable params: 731,584 (2.79 MB)

Epoch 1/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 75s 87ms/step - accuracy: 0.5229 - loss: 1.4501 - val_accuracy: 0.7876 - val_loss: 0.6024
Epoch 2/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 47s 67ms/step - accuracy: 0.8383 - loss: 0.4919 - val_accuracy: 0.8440 - val_loss: 0.4431
Epoch 3/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 46s 66ms/step - accuracy: 0.8757 - loss: 0.3724 - val_accuracy: 0.8672 - val_loss: 0.3760
Epoch 4/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 47s 66ms/step - accuracy: 0.8954 - loss: 0.3153 - val_accuracy: 0.8828 - val_loss: 0.3383
Epoch 5/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 47s 66ms/step - accuracy: 0.9114 - loss: 0.2661 - val_accuracy: 0.8882 - val_loss: 0.3205
MobileNet (fine-tuned) test accuracy: 0.8895000219345093
MobileNet (fine-tuned) test loss    : 0.3229078948497772


In [27]:

# Print the total number of layers inside the ResNet50V2 backbone
print("Total layers in mobilenet_base backbone:", len(mobilenet_base.layers))

# Filter only layers that actually have learnable parameters (weights/biases)
trainable_layers = [layer for layer in mobilenet_base.layers if layer.count_params() > 0]
    
# Print the number of layers that contain learnable parameters "Depth of the Model"
print("Layers with learnable parameters (depth):", len(trainable_layers))


Total layers in mobilenet_base backbone: 154
Layers with learnable parameters (depth): 104


1. Which model achieved the highest accuracy?

MobileNetV2  achieved a test accuracy of ~89%, which is higher than the Custom CNN, which achieved test accuracy of 70.28%. So, MobileNetV2 had the highest accuracy.

2. Which model trained faster?

MobileNetV2 took 47 seconds per epoch in the fine-tuning process (for 5 epochs).
Custom CNN took 12 seconds per epoch on average for 10 epochs.
So, Custom CNN trained faster than MobileNetV2.

3. How might the architecture explain the differences?

MobileNetV2: It is designed to be efficient with depthwise separable convolutions, reducing computational costs while maintaining high accuracy. It is a more complex model compared to the custom CNN, leading to better generalization and performance on the CIFAR-10 dataset. However, this complexity makes it slower to train.

Custom CNN: A simpler model with fewer layers and parameters, making it much faster to train. The simplicity of the architecture means it can't capture as complex patterns or features from the data, leading to lower performance.